# Notebook 2 — PII & Secret Protection (Solo Exercise)
### Module: Enterprise AI Security & Guardrails · 2 of 5

In Notebook 1 you stopped attackers from making BranchLine *say* things it
shouldn't. This notebook is about a quieter failure: BranchLine leaking
personal data **without any attacker at all** — just because a customer or
employee record flowed through it and nobody masked it.

Two places that leak matters:
- **On the way out** — a Social Security number or card number appearing in
  a chat response.
- **On the way in to a vector store** — raw PII getting embedded and indexed,
  where it lives indefinitely and resurfaces in future retrievals (the
  pipeline Notebook 3 builds). Anonymising *before* embedding is the only
  reliable fix; you can't un-embed a number later.

**This is a solo exercise.** Cells marked **🔧 YOUR TURN** have a `# TODO`
for you to complete. The mechanics (finding spans, masking without
corrupting offsets) are given to you in `security_utils/pii.py` — what you
decide is *which* entities to catch and how to *prove* nothing leaks. Run
`pytest tests/test_pii.py -v` to check yourself.


## 0 · Setup

The cell below writes `security_utils/pii.py` to disk — the detection/masking
*mechanics* (finding spans, resolving overlaps, masking right-to-left without
corrupting offsets). That plumbing is given to you; Sections 2-3 are where
you build the part that's actually your call. It exposes a recognizer that
prefers Microsoft Presidio (NER + patterns) and silently falls back to a
regex-only backend if Presidio's spaCy model isn't installed — so this
notebook runs anywhere. Check which backend you got: if it says `regex`,
names and addresses won't be detected (only structured IDs), which is
itself a lesson in why you don't rely on regex alone for PII.

In [ ]:
%%writefile security_utils/pii.py
"""
security_utils/pii.py

Minimal PII detection/masking engine shared by Notebook 2 and its tests.

Design note (why this is "infrastructure" and not something you write):
the *mechanics* of finding spans, resolving overlaps, and masking
right-to-left are fiddly and easy to get subtly wrong (off-by-one indices,
overlapping matches, masking left-to-right and corrupting later offsets).
Those mechanics are given to you here. What you build in the notebook is the
part that actually matters for security: which entity types you choose to
detect, and how you *prove* (with a test) that none of them leak.

Two detection backends:

  * Presidio (preferred) -- Microsoft's PII analyzer, which adds NER-based
    detection of names, locations, etc. on top of pattern matching. Requires
    `presidio-analyzer` + a spaCy model.
  * Regex fallback -- a small, dependency-free set of patterns for the
    structured identifiers that matter most (email, SSN, phone, card, IP).
    Always available, so the notebook runs even where Presidio's model can't
    be installed.

`Recognizer.from_environment()` picks Presidio if it imports and builds, and
silently falls back to regex otherwise.
"""

from __future__ import annotations

import re
from dataclasses import dataclass


@dataclass(frozen=True)
class PiiSpan:
    """One detected PII span: a labelled, half-open [start, end) range."""

    entity_type: str
    start: int
    end: int

    @property
    def length(self) -> int:
        return self.end - self.start


# ---------------------------------------------------------------------------
# Regex backend -- the dependency-free fallback. These cover the structured
# identifiers; names/addresses need Presidio's NER. The notebook discusses
# why regex alone is a floor, not a ceiling.
# ---------------------------------------------------------------------------
DEFAULT_PII_PATTERNS: dict[str, re.Pattern] = {
    # Order matters only for tie-breaking overlaps; SSN before PHONE so a
    # NNN-NN-NNNN string is labelled as an SSN, not a partial phone match.
    "US_SSN": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "EMAIL_ADDRESS": re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
    "CREDIT_CARD": re.compile(r"\b(?:\d[ -]*?){13,16}\b"),
    "PHONE_NUMBER": re.compile(r"\b(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"),
    "IP_ADDRESS": re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
}


def _resolve_overlaps(spans: list[PiiSpan]) -> list[PiiSpan]:
    """Keep non-overlapping spans, preferring earlier-starting then longer
    matches. Masking overlapping spans would corrupt offsets."""
    spans = sorted(spans, key=lambda s: (s.start, -s.length))
    resolved: list[PiiSpan] = []
    last_end = -1
    for s in spans:
        if s.start >= last_end:
            resolved.append(s)
            last_end = s.end
    return resolved


class RegexRecognizer:
    """Pattern-only PII detection. Always available, no model download."""

    backend = "regex"

    def __init__(self, patterns: dict[str, re.Pattern] | None = None):
        self.patterns = patterns or DEFAULT_PII_PATTERNS

    def detect(self, text: str) -> list[PiiSpan]:
        spans = [
            PiiSpan(label, m.start(), m.end())
            for label, pat in self.patterns.items()
            for m in pat.finditer(text)
        ]
        return _resolve_overlaps(spans)


class PresidioRecognizer:
    """NER + pattern PII detection via Microsoft Presidio (if available)."""

    backend = "presidio"

    def __init__(self) -> None:
        from presidio_analyzer import AnalyzerEngine

        self._engine = AnalyzerEngine()

    def detect(self, text: str) -> list[PiiSpan]:
        results = self._engine.analyze(text=text, language="en")
        spans = [PiiSpan(r.entity_type, r.start, r.end) for r in results]
        return _resolve_overlaps(spans)


def get_recognizer(prefer_presidio: bool = True):
    """Return a Presidio recognizer if it imports and builds, else regex.

    Building Presidio can fail if the spaCy model isn't installed; we catch
    that and fall back rather than crashing the notebook.
    """
    if prefer_presidio:
        try:
            return PresidioRecognizer()
        except BaseException:
            # Broad on purpose: a missing spaCy model can surface as an
            # ImportError, an OSError, or even a SystemExit from Presidio's
            # auto-download attempt. Any failure to build means "use regex".
            pass
    return RegexRecognizer()


def mask_text(text: str, spans: list[PiiSpan]) -> str:
    """Replace each span with a ``<ENTITY_TYPE>`` placeholder.

    Applied right-to-left so that masking one span never shifts the offsets
    of spans still to be processed -- the single most common bug when people
    write this themselves.
    """
    for s in sorted(spans, key=lambda s: s.start, reverse=True):
        text = text[: s.start] + f"<{s.entity_type}>" + text[s.end :]
    return text

In [ ]:
import sys, os, json, re
sys.path.insert(0, os.getcwd())

from security_utils.pii import get_recognizer, mask_text, PiiSpan, DEFAULT_PII_PATTERNS
from security_utils.logging_utils import AuditLogger

recognizer = get_recognizer()
audit = AuditLogger(path="logs/pii_audit.jsonl")

print(f"PII backend: {recognizer.backend}")
print(f"Default regex entity types: {list(DEFAULT_PII_PATTERNS)}")

## 1 · See the leak

A realistic helpdesk record. Watch what an unprotected pipeline would put
into a chat response or an embedding.

In [ ]:
EMPLOYEE_RECORD = (
    "Ticket #5518 from Renee Castillo (renee.castillo@millbrookbank.com). "
    "Verified identity with SSN 123-45-6789. Callback number 555-123-4567. "
    "Corporate card on file 4111 1111 1111 1111. "
    "Last login from 10.0.0.42."
)
print(EMPLOYEE_RECORD)
print("\nDetected spans:")
for s in recognizer.detect(EMPLOYEE_RECORD):
    print(f"  {s.entity_type:14s} [{s.start}:{s.end}] -> {EMPLOYEE_RECORD[s.start:s.end]!r}")

## 2 · 🔧 YOUR TURN — mask before exposure

Write `redact()` so it returns the record with **every** detected PII span
replaced by its `<ENTITY_TYPE>` placeholder. You already have the two pieces
you need: `recognizer.detect(text)` and `mask_text(text, spans)`.

In [ ]:
def redact(text: str) -> str:
    # TODO: detect PII spans in `text`, then return the masked version.
    # Hint: this is two function calls.
    raise NotImplementedError

# --- check it ---
masked = redact(EMPLOYEE_RECORD)
print(masked)
assert "123-45-6789" not in masked, "SSN leaked!"
assert "renee.castillo@millbrookbank.com" not in masked, "email leaked!"
assert "4111 1111 1111 1111" not in masked, "card leaked!"
print("\n✅ no structured PII survived")

## 3 · 🔧 YOUR TURN — measure redaction *coverage*

"It looked masked" is not a security guarantee. Coverage = the fraction of
PII spans that actually got masked. Write `coverage()` to return a number in
`[0.0, 1.0]`: of all spans the recognizer finds in the original text, how
many no longer appear as raw substrings in the redacted output?

This is the metric `pytest tests/test_pii.py` asserts on — the exercise's
real deliverable.

In [ ]:
def coverage(original: str, redacted: str) -> float:
    spans = recognizer.detect(original)
    if not spans:
        return 1.0
    # TODO: count how many of the detected raw substrings are GONE from
    # `redacted`, and divide by the total number of spans.
    raise NotImplementedError

cov = coverage(EMPLOYEE_RECORD, redact(EMPLOYEE_RECORD))
print(f"Redaction coverage: {cov:.0%}")
assert cov == 1.0, "Some PII slipped through — fix redact() or your patterns."
print("✅ full coverage")

## 4 · Anonymise before embedding

The critical ordering rule: **mask first, embed second.** Here's a tiny
simulation of an ingestion step — the only correct version redacts before
the text would ever reach `embed()`.

In [ ]:
def embed(text: str) -> list[float]:
    # Stand-in for a real embedding model. The point isn't the vector --
    # it's that whatever string we pass here gets stored forever.
    return [float(len(text))]

def ingest_document(text: str) -> dict:
    safe_text = redact(text)            # <-- mask BEFORE embedding
    audit.log(event="pii_redaction", session_id="ingest",
              verdict="redacted",
              detail={"n_spans": len(recognizer.detect(text))})
    return {"stored_text": safe_text, "vector": embed(safe_text)}

record = ingest_document(EMPLOYEE_RECORD)
print("stored_text:", record["stored_text"])
assert "123-45-6789" not in record["stored_text"]
print("\n✅ what got stored carries no raw SSN")

## 5 · 🔧 YOUR TURN (stretch) — add a custom entity

The regex backend doesn't know about Millbrook Bank's internal employee IDs,
which look like `MB-` followed by 5 digits (e.g. `MB-05518`). Add a
pattern so they get masked too. Then confirm coverage stays at 100%.

In [ ]:
# TODO: add an "EMPLOYEE_ID" entry to this dict matching MB-#####
CUSTOM_PATTERNS = dict(DEFAULT_PII_PATTERNS)
# CUSTOM_PATTERNS["EMPLOYEE_ID"] = re.compile(r"...")

from security_utils.pii import RegexRecognizer
custom = RegexRecognizer(patterns=CUSTOM_PATTERNS)

sample = "Escalating ticket for MB-05518 (renee.castillo@millbrookbank.com)."
spans = custom.detect(sample)
masked = mask_text(sample, spans)
print(masked)
assert "MB-05518" not in masked, "employee ID leaked — add the pattern above"
print("✅ custom entity masked")

## 5b · Graduate your solution so pytest can check it

Once the asserts above pass, write your two functions to a module. The test
suite imports from here — same "notebook prototype → real module" pattern as
Notebook 1. (If you skip this, `pytest tests/test_pii.py` skips with a
reminder.)

In [ ]:
%%writefile security_utils/pii_solution.py
"""Your completed NB2 redaction logic. Written from the notebook."""
from security_utils.pii import get_recognizer, mask_text

_recognizer = get_recognizer()


def redact(text: str) -> str:
    return mask_text(text, _recognizer.detect(text))


def coverage(original: str, redacted: str) -> float:
    spans = _recognizer.detect(original)
    if not spans:
        return 1.0
    gone = sum(1 for s in spans if original[s.start:s.end] not in redacted)
    return gone / len(spans)

In [ ]:
%%writefile tests/test_pii.py
"""
tests/test_pii.py — validates the PII redaction logic you complete in
Notebook 2. Skips with a reminder until you run the notebook's Section 5b
`%%writefile` cell, which graduates your redact()/coverage() into
security_utils/pii_solution.py.
"""
import re

import pytest

from security_utils.pii import DEFAULT_PII_PATTERNS, RegexRecognizer, mask_text

try:
    from security_utils.pii_solution import coverage, redact
except ModuleNotFoundError:
    pytest.skip(
        "security_utils/pii_solution.py not found. Complete Notebook 2 and "
        "run its Section 5b cell to graduate your solution first.",
        allow_module_level=True,
    )

RECORD = (
    "Ticket #5518 from Renee Castillo (renee.castillo@millbrookbank.com). "
    "Verified identity with SSN 123-45-6789. Callback number 555-123-4567. "
    "Corporate card on file 4111 1111 1111 1111. Last login from 10.0.0.42."
)


@pytest.mark.parametrize(
    "raw", ["123-45-6789", "renee.castillo@millbrookbank.com", "4111 1111 1111 1111"]
)
def test_structured_pii_is_removed(raw):
    assert raw not in redact(RECORD)


def test_full_coverage_on_record():
    assert coverage(RECORD, redact(RECORD)) == 1.0


def test_clean_text_is_unchanged_and_full_coverage():
    clean = "Remote work eligibility requires 12 months tenure."
    assert redact(clean) == clean
    assert coverage(clean, redact(clean)) == 1.0


def test_custom_employee_id_pattern_masks():
    """The stretch goal: a custom MB-##### entity should be maskable."""
    patterns = dict(DEFAULT_PII_PATTERNS)
    patterns["EMPLOYEE_ID"] = re.compile(r"\bMB-\d{5}\b")
    rr = RegexRecognizer(patterns=patterns)
    sample = "Escalating ticket for MB-05518."
    assert "MB-05518" not in mask_text(sample, rr.detect(sample))

## 6 · Audit trail & wrap-up

Every redaction was logged to `logs/pii_audit.jsonl` — a durable record that
ingestion *happened*, without storing the PII itself (the log keeps a count,
never the values). In a real LangSmith setup you'd also tag the trace; here
the local audit log is the same idea, vendor-independent.

```bash
pytest tests/test_pii.py -v
```

**Next — Notebook 3 (Guided Lab): RAG Poisoning.** You'll build the retrieval
pipeline these redacted documents flow into, then watch an attacker poison it
with an adversarial document, and add source-validation guardrails.

In [ ]:
print(f"{len(audit.read_all())} redaction events logged to logs/pii_audit.jsonl")